# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. It demonstrates key steps for working with structured clinical data defined by a Croissant schema and shows how to reference entities by their `@id` values.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant-defined dataset using `mlcroissant`. This section demonstrates how to retrieve the metadata, including dataset name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset Title:", metadata['name'])
print("Description:", metadata['description'])

## 2. Data Overview
Review available record sets, fields, and their IDs. Croissant datasets use the `@id` property to uniquely identify all entities. This step shows how to list the record sets defined in the schema, along with their fields.

In [ ]:
# Inspect Croissant record sets
from pprint import pprint

# Extract record sets from the schema metadata
record_sets = dataset.metadata['recordSet'] if 'recordSet' in dataset.metadata else []

print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            print(f"    - Field @id: {field['@id']} ({field.get('name','')})")
    else:
        print("  No fields defined.")
    print()
# Optionally, show each record of the first record set for demonstration
if record_sets:
    record_set_id = record_sets[0]['@id']
    print(f"Example records from RecordSet @id: {record_set_id}")
    for record in dataset.records(record_set=record_set_id):
        pprint(record)
        break  # show just the first record

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id` values from the overview. Only reference entities by their `@id`.

In [ ]:
# Extract all records from each record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Record sets found: {record_set_ids}")

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:  # Only add if there are records
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id = {rs_id}")
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head(2))

# For demonstration, use the first record set loaded
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    selected_df = dataframes[selected_record_set_id]
    print(f"\nUsing RecordSet @id = {selected_record_set_id} for further analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Fields are referenced strictly by their `@id` from the Croissant schema.

In [ ]:
# Example: Filter records based on numeric field using Croissant @id
# Find numeric fields in the selected record set
rs = next((rs for rs in record_sets if rs['@id'] == selected_record_set_id), None)
numeric_fields = []
if rs and 'field' in rs:
    for field in rs['field']:
        if field.get('@type') in ['schema:Integer', 'schema:Float', 'schema:Number']:
            numeric_fields.append(field['@id'])

print(f"Numeric fields (@id): {numeric_fields}")

# Choose a numeric field for demonstration
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field @id: {numeric_field_id}")
    # Use a threshold for filtering
    threshold = 10
    # Ensure that field exists and is numeric
    filtered_df = selected_df[selected_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Choose a group field: Find a categorical field
    group_fields = []
    for field in rs['field']:
        if field.get('@type') in ['schema:Text']:
            group_fields.append(field['@id'])

    if group_fields:
        group_field_id = group_fields[0]
        print(f"Grouping by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print("Grouped data (mean of numeric field):")
        print(grouped_df.head())
else:
    print("No numeric fields found to demonstrate filtering and normalization.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using pandas and matplotlib. Visualizations are based on fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution if numeric fields exist
if dataframes and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(selected_df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_fields:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=selected_df[group_field_id], y=selected_df[numeric_field_id])
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated the use of the `mlcroissant` library for loading the FAIR^2 dataset and referencing all entities by their Croissant `@id`. Common steps included metadata access, record set extraction, filtering and normalization of numeric data, and basic visualizations. This approach supports transparent, reproducible analysis and prepares the dataset for downstream clinical or statistical modeling tasks.

*End of notebook*